In [1]:
import pyro
import torch

import matplotlib.pyplot as plt

from mmbo.nn import RegBNN, RegBNNGuide, LayeredRegBNN, LayeredRegBNNGuide, posterior_samples
from mmbo.examples.demo import paciorek_demo, branin_demo
from mmbo.examples.testfunc import branin_hf
from mmbo.training import do_svi

#device = "cuda:3"

## Get training data

In [2]:
X_train_main = branin_demo['X_train_main']
X_train_mm = branin_demo['X_train_mm']
X_test = branin_demo['X_test']
Y_train_main = branin_demo['Y_train_main']
Y_train_mm = branin_demo['Y_train_mm']
Y_test_main = branin_demo['Y_test_main']
Y_test_mm = branin_demo['Y_test_mm']

## Train uni-modal model

In [3]:
# model = RegBNN.from_data(X_train = X_train_main, Y_train = Y_train_main, hidden_widths=[256,256])
# guide = RegBNNGuide(model)

# #model = model.to(device)
# #guide = guide.to(device)

In [4]:
# adam = pyro.optim.Adam({"lr": 0.001})
# elbo = pyro.infer.Trace_ELBO(num_particles = 1)

# svi = pyro.infer.SVI(model, guide, adam, elbo)
# params, losses = do_svi(svi, params = None)

In [5]:
# plt.plot(losses)
# plt.yscale("log")

In [6]:
# samples = posterior_samples(guide, params, 1000)

## Train multi-modal layered model

In [7]:
# model_layered = LayeredRegBNN.from_data(X_train_main = X_train_main, Y_train_main = Y_train_main, X_train_aux = X_train_mm, Y_train_aux = Y_train_mm, hidden_widths=[256,256])
# guide_layered = LayeredRegBNNGuide(model_layered)

In [8]:
# adam_layered = pyro.optim.Adam({"lr": 0.001})
# elbo_layered = pyro.infer.Trace_ELBO(num_particles = 1)

# svi_layered = pyro.infer.SVI(model_layered, guide_layered, adam_layered, elbo_layered)
# params_layered, losses_layered = do_svi(svi_layered, params = None)

In [9]:
# samples_layered = posterior_samples(guide_layered, params_layered, 1000)

## Use botorch optimizer on fitted models

## Optimize expected improvement

In [10]:
from mmbo.optim import expected_improvement, lcb, probability_improvement, probability_improvement_sigmoid, expected_improvement_softplus, expected_improvement_softplus_batch
from mmbo.examples.testfunc import paciorek_hf, branin_hf
import botorch
import datetime

In [11]:
n_raw_samples = 1000

low_lim = torch.tensor([-5.0, 0.0])
high_lim = torch.tensor([10., 15.0])

# Paciorek
#X_init = torch.rand((n_swarm, 4)) * (high_lim - low_lim) + low_lim
# Branin
X_init = torch.rand((n_raw_samples, 2)) * (high_lim - low_lim) + low_lim

In [ ]:
def rect_sample(n: int, low_lim: torch.Tensor, high_lim: torch.Tensor) -> torch.Tensor:
    assert low_lim.shape == high_lim.shape
    return torch.rand((n, *low_lim.shape)) * (high_lim - low_lim) + low_lim

def select_next_point(objective, num_candidates, low_lim, high_lim, num_raw_samples=None, initial_samples=None):
    if num_raw_samples is None and initial_samples is None:
        raise ValueError("Must supply either num_raw_samples or initial_samples")
    if num_raw_samples is not None and initial_samples is not None:
        raise Warning("num_raw_samples is ignored if initial_samples is given")
    if initial_samples is None:
        initial_samples = rect_sample(num_raw_samples, low_lim, high_lim)
    # Select the top 'num_candidates' of the initial samples to use as starting points
    raw_objective = objective(initial_samples)
    _,bestidx = torch.topk(raw_objective.squeeze(), num_candidates)
    candidates = initial_samples[bestidx]
    # Use the botorch SLSQP optimizer and select the best final position
    final_samples,final_objective = botorch.gen_candidates_scipy(candidates, objective, low_lim, high_lim, options={"method":"SLSQP"})
    _,finalidx = torch.topk(final_objective.squeeze(), 1)
    return final_samples[finalidx]

### Uni-modal

In [13]:


#objective = lambda x: expected_improvement_softplus(model, samples, x, Y_train_main.min())

#num_points = 50

#X_new = select_next_point(objective, num_points, low_lim, high_lim, initial_samples = X_init)
#print(X_new)


### Layered Multi-modal

In [14]:
#objective_layered = lambda x: expected_improvement_softplus(model_layered, samples_layered, x, Y_train_main.min())

#X_new_layered = select_next_point(objective_layered, num_points, low_lim, high_lim, initial_samples = X_init)
#print(X_new_layered)

## Loop, adding new points and keeping track of reward

### Uni-modal

In [16]:
n_search = 10

params = None
num_points = 50
num_raw_samples = 1000

Y_train_main_loop = Y_train_main.clone()
X_train_main_loop = X_train_main.clone()

running_min = [Y_train_main_loop.min().item()]

for iter in range(n_search):
    # Rebuild and Train the model
    model = RegBNN.from_data(X_train = X_train_main_loop, Y_train = Y_train_main_loop, hidden_widths=[256,256])
    guide = RegBNNGuide(model)
    adam = pyro.optim.Adam({"lr": 0.001})
    elbo = pyro.infer.Trace_ELBO(num_particles = 1)
    svi = pyro.infer.SVI(model, guide, adam, elbo)
    model.train()
    guide.train()
    params,_ = do_svi(svi, params = params, check_interval = 500)
    samples = posterior_samples(guide, params, 1000)
    model.eval()
    guide.eval()
    # Which new point to select?
    #print(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Start unbatched search")
    #objective = lambda x: expected_improvement_softplus(model, samples, x, Y_train_main_loop.min())
    #X_new = select_next_point(objective, num_points, low_lim, high_lim, num_raw_samples = num_raw_samples)
    print(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Start batched search")
    objective = lambda x: expected_improvement_softplus_batch(model, samples, x, Y_train_main_loop.min())
    X_new_batch = select_next_point(objective, num_points, low_lim, high_lim, num_raw_samples = num_raw_samples)
    print(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - End batched search")
    # "Run experiment" and append to training data
    Y_new = branin_hf(X_new_batch)
    Y_train_main_loop = torch.cat([Y_train_main_loop, Y_new], dim=0)
    X_train_main_loop = torch.cat([X_train_main_loop, X_new_batch], dim=0)
    # Print everything, track attained minimum
    running_min.append(Y_train_main_loop.min().item())
    print(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {iter}: f({X_new_batch}) = {Y_new}. Min = {Y_train_main_loop.min().item()}")

2025-10-08 13:04:23 - Start


/home/itaylor/projects/mmbo/.venv/lib64/python3.9/site-packages/torch/distributions/wishart.py:275: UserWarning: Singular sample detected.
  warnings.warn("Singular sample detected.")


2025-10-08 13:04:38 - Step 500, Elbo loss: 60255.1 +/- 18617.8 (slope = -64276.000 +/- 398.368 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:04:52 - Step 1000, Elbo loss: 35086.7 +/- 11043.3 (slope = -38112.931 +/- 252.700 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:05:06 - Step 1500, Elbo loss: 20245.3 +/- 6448.5 (slope = -22243.284 +/- 161.283 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:05:22 - Step 2000, Elbo loss: 11634.0 +/- 3723.4 (slope = -12840.430 +/- 96.386 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:05:38 - Step 2500, Elbo loss: 6678.6 +/- 2135.9 (slope = -7353.907 +/- 66.708 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:05:53 - Step 3000, Elbo loss: 3843.0 +/- 1217.2 (slope = -4187.253 +/- 41.140 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:06:08 - Step 3500, Elbo loss: 2225.4 +/- 698.0 (slope = -2393.286 +/- 29.281 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:06:24 - Step 4000, Elbo loss: 1306.0 +/- 405.3 (slope = -1381.467 +/- 21.718 loss/k-iter, p-value = 

In [ ]:
# {n:samples[n].shape for n in samples}

### Multi-modal Layered

In [17]:
n_search = 10

params_layered = None
num_points = 50
num_raw_samples = 1000

Y_train_main_loop_layered = Y_train_main.clone()
X_train_main_loop_layered = X_train_main.clone()

running_min_layered = [Y_train_main_loop_layered.min().item()]

for iter in range(n_search):
    # Rebuild and Train the model
    model = LayeredRegBNN.from_data(X_train_main = X_train_main_loop_layered, Y_train_main = Y_train_main_loop_layered, X_train_aux = X_train_mm, Y_train_aux = Y_train_mm, hidden_widths=[256,256])
    guide = LayeredRegBNNGuide(model)
    adam = pyro.optim.Adam({"lr": 0.001})
    elbo = pyro.infer.Trace_ELBO(num_particles = 1)
    svi = pyro.infer.SVI(model, guide, adam, elbo)
    model.train()
    guide.train()
    params_layered,_ = do_svi(svi, params = params_layered, check_interval = 500)
    samples = posterior_samples(guide, params_layered, 1000)
    model.eval()
    guide.eval()
    # Which new point to select?
    #print(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Start unbatched search")
    #objective = lambda x: expected_improvement_softplus(model, samples, x, Y_train_main_loop_layered.min())
    #X_new = select_next_point(objective, num_points, low_lim, high_lim, num_raw_samples = num_raw_samples)
    print(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Start batched search")
    objective = lambda x: expected_improvement_softplus_batch(model, samples, x, Y_train_main_loop_layered.min())
    X_new_batch = select_next_point(objective, num_points, low_lim, high_lim, num_raw_samples = num_raw_samples)
    print(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - End batched search")
    # "Run experiment" and append to training data
    Y_new = branin_hf(X_new_batch)
    Y_train_main_loop_layered = torch.cat([Y_train_main_loop_layered, Y_new], dim=0)
    X_train_main_loop_layered = torch.cat([X_train_main_loop_layered, X_new_batch], dim=0)
    # Print everything, track attained minimum
    running_min_layered.append(Y_train_main_loop_layered.min().item())
    print(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {iter}: f({X_new_batch}) = {Y_new}. Min = {Y_train_main_loop_layered.min().item()}")

2025-10-08 13:29:37 - Start
2025-10-08 13:30:03 - Step 500, Elbo loss: 60018.6 +/- 18535.5 (slope = -63993.523 +/- 394.656 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:30:28 - Step 1000, Elbo loss: 35026.8 +/- 10951.0 (slope = -37792.098 +/- 253.053 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:30:52 - Step 1500, Elbo loss: 20306.1 +/- 6417.7 (slope = -22136.783 +/- 160.976 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:31:16 - Step 2000, Elbo loss: 11735.1 +/- 3708.7 (slope = -12786.016 +/- 99.925 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:31:40 - Step 2500, Elbo loss: 6775.8 +/- 2147.3 (slope = -7392.177 +/- 67.999 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:32:05 - Step 3000, Elbo loss: 3912.4 +/- 1236.2 (slope = -4246.710 +/- 46.360 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:32:29 - Step 3500, Elbo loss: 2268.7 +/- 711.1 (slope = -2437.417 +/- 30.347 loss/k-iter, p-value = 0.000) + 
2025-10-08 13:32:53 - Step 4000, Elbo loss: 1318.8 +/- 415.6 (slope = -1415.433 +/- 22